# Выполнение ЛР №3: Введение в построение признаков

## Подключение библиотек

In [ ]:
import math
import re

import pandas               as pd
import numpy                as np

import matplotlib           as mpl
import matplotlib.pyplot    as mpl_plt

from sklearn.preprocessing import LabelEncoder

## Настройка библиотек

In [ ]:
mpl_plt.show()

pd.set_option('display.max_rows', None)

## Задание 1

### Формулировка

* Изучить  датасет и контекст (при наличии) в папке с 
вариантом:  определить  типы  данных  признаков  и 
назначение

### Решение

In [ ]:
from pathlib import Path

DATA_PATH = Path('Вариант 4') / 'OnlineRetail.csv'

print(f'Путь к датасету: "{DATA_PATH.resolve()}"')
print(f'Размер файла: {DATA_PATH.stat().st_size / 1024:.1f} КБ')

retail_df = pd.read_csv(DATA_PATH, encoding='latin1')

print('\nПервые строки:')
display(retail_df.head())

print('\nИнформация о столбцах:')
retail_df.info()

display(retail_df.describe(include=['int64', 'float64']))
display(retail_df.describe(include=['object']))

### Замечание

Обнаружено несоответствие между описанием в [context.md](./Вариант%204/context.md) (геометрия зерен пшеницы) и фактическим датасетом транзакций;

### Расшифровка признаков датасета

#### InvoiceNo (Номер счета-фактуры)

- **Тип признака:** категориальный
- **Назначение:**   номер счета-фактуры, объединяет несколько строк одной покупки.

* * *

#### StockCode (Каталожный артикул)

- **Тип признака:** категориальный
- **Назначение:**   каталожный артикул, используется для группировки и агрегаций по товару.

* * *

#### Quantity (Количество)

- **Тип признака:** числовой
- **Назначение:**   количество единиц товара в строке транзакции, используется для расчета объемов продаж.

* * *

#### InvoiceDate (Дата формирования счета)

- **Тип признака:** числовой 
- **Назначение:**   дата и время оформления счёта, источник временных признаков.

* * *

#### UnitPrice (Цена)

- **Тип признака:** числовой
- **Назначение:**   цена за единицу товара, участвует в расчете выручки.

* * *

#### CustomerID (Идентификатор клиента)

- **Тип признака:** категориальный
- **Назначение:**   уникальный идентификатор клиента, используется для сегментации и RFM-анализа.

* * *

#### Country (Страна)

- **Тип признака:** категориальный
- **Назначение:** страна клиента, применяется в географическом анализе.

## Задание 2

### Формулировка

* Обнаружить  и  обработать выбросы  в  значениях 
признаков. 

### Решение

#### Визуализация: графики признаков по номеру строки

Построим для каждого столбца график значения от индекса строки (номер строки = ось X).
Чтобы не перегружать вывод при большом объёме данных, используем семплирование до `max_points` точек.


In [ ]:

# Параметр: максимальное число точек для рисования (измените при необходимости)
max_points = 5000

retail_df_visualization = retail_df.copy()
n = len(retail_df_visualization)
print(f'Всего строк в датасете: {n}')

# Если слишком много строк, равномерно семплируем индексы, чтобы увидеть общую структуру
if n > max_points:
    sample_idx = np.linspace(0, n - 1, max_points, dtype=int)
    sampled = retail_df_visualization.iloc[sample_idx].reset_index(drop=True)
    x_vals = sample_idx
    print(f'Семплировано {max_points} точек (равномерно).')
else:
    sampled = retail_df_visualization.reset_index(drop=True)
    x_vals = sampled.index.values

# Попробуем корректно распарсить дату, если есть столбец InvoiceDate
if 'InvoiceDate' in sampled.columns:
    sampled['InvoiceDate_parsed'] = pd.to_datetime(sampled['InvoiceDate'], errors='coerce')

cols = [c for c in sampled.columns if not c.startswith('InvoiceDate_parsed')]
n_cols = len(cols)

# Список признаков, требующих внимания — укажите свои
attention_cols = {'Quantity', 'UnitPrice'}  # <- настройте

# Цвет подсветки и параметры
highlight_bg = '#fff0f0'   # светло-розовый фон для тревожных графиков
highlight_edge = 'red'
annotation_text = 'обнаружены выбросы'

# Настройка сетки под графики: 2 столбца, строки по числу признаков
ncols_plot = 2
nrows_plot = math.ceil(n_cols / ncols_plot)
figsize = (14, 3 * nrows_plot)
fig, axes = mpl_plt.subplots(nrows_plot, ncols_plot, figsize=figsize, constrained_layout=True)
axes = axes.flatten()

for i, col in enumerate(cols):
    ax = axes[i]
    ser = sampled[col]
    
    # Если признак помечен для внимания — меняем фон и рамку
    is_attention = col in attention_cols
    if is_attention:
        ax.set_facecolor(highlight_bg)
        for spine in ax.spines.values():
            spine.set_edgecolor(highlight_edge)
            spine.set_linewidth(1.2)

    # Если столбец — дата (мы создали parsed), используем parsed версию
    if col == 'InvoiceDate' and 'InvoiceDate_parsed' in sampled.columns:
        y = sampled['InvoiceDate_parsed']
        # рисуем как точечный график времени по номеру строки
        ax.scatter(x_vals, y, s=6, alpha=0.6)
        ax.set_ylabel('Дата/время')
    else:
        # Для числовых типов — прямая отрисовка, для категорий — факторизация
        if pd.api.types.is_numeric_dtype(ser):
            ax.plot(x_vals, ser.values, linestyle='-', marker='.', markersize=2, alpha=0.6)
            ax.set_ylabel('Значение')
        else:
            # факторизуем категориальные значения в коды для отображения
            codes, uniques = pd.factorize(ser, sort=True)
            # Заменим код -1 (для NaN) на np.nan чтобы не мешался
            codes = np.where(codes == -1, np.nan, codes)
            # Добавим немного шума, чтобы точки не наслаивались полностью
            jitter = np.random.normal(scale=0.08, size=len(codes))
            ax.scatter(x_vals, codes + (jitter if len(codes)==len(jitter) else 0), s=6, alpha=0.6)
            ax.set_ylabel('Категории (код)')
            # Если немного уникальных значений, подпишем ось Y метками
            if len(uniques) <= 20:
                ticks = np.arange(len(uniques))
                ax.set_yticks(ticks)
                ax.set_yticklabels([str(u) for u in uniques], fontsize=8)
    
    title = f'{col}'
    if is_attention:
        title += f' - ({annotation_text})'
    ax.set_title(title)
    ax.set_xlabel('Номер строки (индекс)')

# Если графиков меньше ячеек, уберём пустые
for j in range(n_cols, len(axes)):
    try:
        fig.delaxes(axes[j])
    except Exception:
        pass

mpl_plt.suptitle('Графики признаков по номеру строки (семплирование при необходимости)', fontsize=14)
mpl_plt.show()

# Дополнительно: вывести распределения/базовую статистику по каждому признаку (коротко)
print('Краткая статистика по признакам:')
for col in cols:
    ser = sampled[col]
    if pd.api.types.is_numeric_dtype(ser):    
        print(f'- {col}: mean={ser.mean():.3f}, std={ser.std():.3f}, min={ser.min()}, max={ser.max()}')
    else:        
        print(f'- {col}: уникальных={ser.nunique(dropna=True)}, примеры={ser.dropna().unique()[:5]}') 

del retail_df_visualization

#### Подготовка данных

In [ ]:
# Создаем копию датасета для работы с выбросами
retail_df_outliers = retail_df.copy()

# Определяем признаки для анализа
numeric_cols = ['Quantity', 'UnitPrice']

print("\n1. Предварительный анализ числовых признаков:")
print("-" * 60)
for col in numeric_cols:
    print(f"\n{col}:")
    print(f"  Минимум: {retail_df_outliers[col].min()}")
    print(f"  Максимум: {retail_df_outliers[col].max()}")
    print(f"  Среднее: {retail_df_outliers[col].mean():.2f}")
    print(f"  Медиана: {retail_df_outliers[col].median():.2f}")
    print(f"  Стандартное отклонение: {retail_df_outliers[col].std():.2f}")
    print(f"  Отрицательных значений: {(retail_df_outliers[col] < 0).sum()}")
    print(f"  Нулевых значений: {(retail_df_outliers[col] == 0).sum()}")

# Статистика по выбросам
display(retail_df_outliers[numeric_cols].describe())

#### Удаление отрицательных значений

In [ ]:
# Определяем числовые признаки для проверки
numeric_cols = ['Quantity', 'UnitPrice']

# Вычисляем медианные значения для каждого признака (только для положительных значений)
print("\nМедианные значения (по положительным значениям):")
medians = {}
for col in numeric_cols:
    # Вычисляем медиану только для положительных значений
    positive_values = retail_df_outliers[retail_df_outliers[col] >= 0][col]
    median_val = positive_values.median()
    medians[col] = median_val
    print(f"  {col}: медиана = {median_val:.2f}")

# Подсчитываем отрицательные значения по каждому признаку до замены
print("\nОтрицательные значения по признакам (до замены):")
for col in numeric_cols:
    negative_count = (retail_df_outliers[col] < 0).sum()
    print(f"  {col}: {negative_count} отрицательных значений")

# Заменяем отрицательные значения на медианные
print("\nВыполняется замена отрицательных значений...")
for col in numeric_cols:
    mask_negative = retail_df_outliers[col] < 0
    negative_count = mask_negative.sum()
    if negative_count > 0:
        retail_df_outliers.loc[mask_negative, col] = medians[col]
        print(f"  {col}: заменено {negative_count} значений на медиану ({medians[col]:.2f})")

# Проверяем результат
print("\nПроверка после замены:")
print("-" * 60)
for col in numeric_cols:
    negative_count = (retail_df_outliers[col] < 0).sum()
    print(f"  {col}: отрицательных значений = {negative_count}")
    print(f"    Минимум: {retail_df_outliers[col].min()}")
    print(f"    Максимум: {retail_df_outliers[col].max()}")
    print(f"    Медиана: {retail_df_outliers[col].median():.2f}")

# Выводим обновленную статистику
print("\nОбновленная статистика по числовым признакам:")
display(retail_df_outliers[numeric_cols].describe())


#### Ограничение положительных выбросов по процентилям


In [ ]:
print("\nВычисление процентилей и медианы:")
print("-" * 60)
percentiles = {}
medians = {}

for col in numeric_cols:
    # Вычисляем 95% процентили
    p95 = retail_df_outliers[col].quantile(0.95)
    median_val = retail_df_outliers[col].median()
    
    percentiles[col] = {'p95': p95}
    medians[col] = median_val
    
    print(f"\n{col}:")
    print(f"  95% процентиль: {p95:.2f}")
    print(f"  Медиана: {median_val:.2f}")

# Подсчитываем выбросы до обработки
print("\nВыбросы до обработки:")
print("-" * 60)
for col in numeric_cols:
    p95 = percentiles[col]['p95']
    
    # Находим значения, выходящие за пределы 95%
    outliers_high = (retail_df_outliers[col] > p95).sum()
    
    print(f"\n{col}:")
    print(f"  Значений > 95% процентиля ({p95:.2f}): {outliers_high}")

# Ограничиваем выбросы
print("\nВыполняется ограничение выбросов...")
print("-" * 60)

for col in numeric_cols:
    p95 = percentiles[col]['p95']
    
    # Для Quantity: заменяем на граничное значение (95% процентиль)
    if col == 'Quantity':
        # Значения больше 95% процентиля заменяем на 95% процентиль
        mask_high = retail_df_outliers[col] > p95
        count_high = mask_high.sum()
        if count_high > 0:
            retail_df_outliers.loc[mask_high, col] = p95
            print(f"  {col}: заменено {count_high} значений > 95% на {p95:.2f}")
    
    # Для UnitPrice: заменяем на медианное значение
    elif col == 'UnitPrice':
        median_val = medians[col]
        
        # Значения больше 95% процентиля заменяем на медиану
        mask_high = retail_df_outliers[col] > p95
        count_high = mask_high.sum()
        if count_high > 0:
            retail_df_outliers.loc[mask_high, col] = median_val
            print(f"  {col}: заменено {count_high} значений > 95% на медиану ({median_val:.2f})")

# Проверяем результат
print("\nПроверка после ограничения выбросов:")
print("=" * 60)
for col in numeric_cols:
    p95 = percentiles[col]['p95']

    outliers_high = (retail_df_outliers[col] > p95).sum()
    
    print(f"\n{col}:")
    print(f"  Значений > 95% процентиля: {outliers_high}")
    print(f"  Минимум: {retail_df_outliers[col].min():.2f}")
    print(f"  Максимум: {retail_df_outliers[col].max():.2f}")
    print(f"  Медиана: {retail_df_outliers[col].median():.2f}")

# Выводим обновленную статистику
print("\nОбновленная статистика по числовым признакам:")
display(retail_df_outliers[numeric_cols].describe())

#### Визуализация обработанных данных

In [ ]:
# Параметр: максимальное число точек для рисования (измените при необходимости)
max_points = 5000

sample_idx = np.linspace(0, n - 1, max_points, dtype=int)
sampled = retail_df_outliers.iloc[sample_idx].reset_index(drop=True)
x_vals = sample_idx
print(f'Семплировано {max_points} точек (равномерно).')

figsize = (10, 3 * len(numeric_cols))
fig, axes = mpl_plt.subplots(len(numeric_cols), 1, figsize=figsize, constrained_layout=True)
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    ser = sampled[col]
    ax.plot(x_vals, ser.values, linestyle='-', marker='.', markersize=2, alpha=0.6)
    ax.set_ylabel('Значение')
    
    ax.grid(axis='y')
    
    ax.set_title(f'{col}')

axes[len(numeric_cols)-1].set_xlabel('Номер строки (индекс)')   
    
    
mpl_plt.suptitle('Графики признаков после обработки выбросов', fontsize=14)
mpl_plt.show()


## Задание 3

### Формулировка

* Обработать  пропущенные  значения, подобрав 
подходящий алгоритм.

### Решение

#### Анализ пропущенных значений

Сначала проанализируем, какие признаки содержат пропущенные значения и в каком количестве.

In [ ]:
print("Анализ пропущенных значений:")
print("=" * 60)

retail_df_empy_vall = retail_df_outliers.copy()

missing_analysis = pd.DataFrame({
    'Количество пропусков': retail_df_empy_vall.isnull().sum(),
    'Процент пропусков': (retail_df_empy_vall.isnull().sum() / len(retail_df_empy_vall) * 100).round(2),
    'Количество заполненных': retail_df_empy_vall.notna().sum()
})
missing_analysis = missing_analysis[missing_analysis['Количество пропусков'] > 0].sort_values('Количество пропусков', ascending=False)

display(missing_analysis)

# Проверим, пересекаются ли пропуски
print("\n2. Анализ пересечения пропусков:")
both_missing = retail_df_empy_vall[retail_df_empy_vall['Description'].isnull() & retail_df_empy_vall['CustomerID'].isnull()]
print(f"   Строк с пропусками в обоих признаках: {len(both_missing)}")

# Примеры строк с пропусками
print("\n3. Примеры строк с пропущенными значениями:")
mask = retail_df_empy_vall.isnull().any(axis=1)
display(retail_df_empy_vall.loc[mask].head(10))


#### Выбор алгоритма обработки пропущенных значений

**Обоснование выбора стратегии:**

1. **Description (Описание товара)** - ~0.27% пропусков:
   - **Стратегия:** Заполнение по StockCode (каталожному артикулу)
   - **Обоснование:** 
     - Описание товара тесно связано с его артикулом (StockCode)
     - Один и тот же артикул должен иметь одинаковое описание
     - Можно использовать наиболее частое описание для данного StockCode
     - Если для StockCode нет описания, используем сам StockCode как описание
   - **Алгоритм:** `fillna()` с группировкой по StockCode и применением `mode()` или `first()`

2. **CustomerID (Идентификатор клиента)** - ~24.9% пропусков:
   - **Стратегия:** Создание специальной категории для анонимных клиентов
   - **Обоснование:**
     - Большой процент пропусков (почти 25%) указывает на систематический характер (возможно, анонимные покупки)
     - CustomerID - категориальный идентификатор, его нельзя интерполировать
     - Заполнение средним/медианой не имеет смысла для идентификатора
     - Создание отдельной категории (например, 0 или -1) позволит сохранить информацию о наличии пропуска
   - **Алгоритм:** `fillna()` с специальным значением (0 или -1) с последующим преобразованием в категориальный тип


In [ ]:
# Создаем копию датасета для обработки
retail_df_empy_vall = retail_df.copy()

print("Обработка пропущенных значений:")
print("=" * 60)

# 1. Обработка Description
print("\n1. Обработка Description (Описание товара):")
print(f"   Пропусков до обработки: {retail_df_empy_vall['Description'].isnull().sum()}")

# Заполняем пропуски Description на основе StockCode
# Для каждого StockCode находим наиболее частое описание
stockcode_to_description = retail_df_empy_vall.groupby('StockCode')['Description'].apply(
    lambda x: x.mode().iloc[0] if not x.mode().empty else None
).to_dict()

# Заполняем пропуски
def fill_description(row):
    if pd.isna(row['Description']):
        stock_code = row['StockCode']
        if stock_code in stockcode_to_description and stockcode_to_description[stock_code] is not None:
            return stockcode_to_description[stock_code]
        else:
            # Если для StockCode нет описания, используем сам StockCode
            return f"Unknown Item ({stock_code})"
    return row['Description']

retail_df_empy_vall['Description'] = retail_df_empy_vall.apply(fill_description, axis=1)

print(f"   Пропусков после обработки: {retail_df_empy_vall['Description'].isnull().sum()}")

# 2. Обработка CustomerID
print("\n2. Обработка CustomerID (Идентификатор клиента):")
print(f"   Пропусков до обработки: {retail_df_empy_vall['CustomerID'].isnull().sum()}")

# Заполняем пропуски специальным значением для анонимных клиентов
# Используем 0 как идентификатор анонимного клиента
retail_df_empy_vall['CustomerID'] = retail_df_empy_vall['CustomerID'].fillna(0)

# Преобразуем в целочисленный тип (так как это идентификатор)
retail_df_empy_vall['CustomerID'] = retail_df_empy_vall['CustomerID'].astype(int)

print(f"   Пропусков после обработки: {retail_df_empy_vall['CustomerID'].isnull().sum()}")

# Проверка результата
print("\n" + "=" * 60)
print("Итоговая проверка пропущенных значений:")
print("=" * 60)
final_missing = retail_df_empy_vall.isnull().sum()
final_missing = final_missing[final_missing > 0]
if len(final_missing) == 0:
    print("✓ Все пропущенные значения успешно обработаны!")
else:
    print("Остались пропуски в следующих признаках:")
    display(final_missing)

print("\nИнформация о датасете после обработки:")
retail_df_empy_vall.info()


#### Визуализация результатов обработки

Проверим качество обработки и покажем примеры заполненных значений.


In [ ]:

# 1. Сравнение до и после обработки
comparison = pd.DataFrame({
    'До обработки': [retail_df['Description'].isnull().sum(), retail_df['CustomerID'].isnull().sum()],
    'После обработки': [retail_df_empy_vall['Description'].isnull().sum(), retail_df_empy_vall['CustomerID'].isnull().sum()]
}, index=['Description', 'CustomerID'])

print("Сравнение количества пропусков до и после обработки:")
display(comparison)

# 2. Примеры заполненных значений Description
print("\nПримеры заполненных значений Description:")
# Найдем строки, где Description был заполнен
original_missing_desc = retail_df['Description'].isnull()
filled_desc = retail_df_empy_vall[original_missing_desc][['StockCode', 'Description']].head(10)
display(filled_desc)

# 3. Статистика по CustomerID
print("\nСтатистика по CustomerID:")
cust_stats = pd.DataFrame({
    'Категория': ['Идентифицированные клиенты', 'Анонимные клиенты (CustomerID=0)'],
    'Количество транзакций': [
        (retail_df_empy_vall['CustomerID'] != 0).sum(),
        (retail_df_empy_vall['CustomerID'] == 0).sum()
    ],
    'Процент': [
        (retail_df_empy_vall['CustomerID'] != 0).sum() / len(retail_df_empy_vall) * 100,
        (retail_df_empy_vall['CustomerID'] == 0).sum() / len(retail_df_empy_vall) * 100
    ]
})
display(cust_stats)

## Задание 4

### Формулировка

* Выполнить  преобразование  категориальных  данных 
(при наличии), подобрав подходящие алгоритмы.

In [ ]:
# Создаем копию датасета для преобразования категориальных данных
retail_df_categorical = retail_df_empy_vall.copy()

print(f"\nИсходная информация о датасете:")
retail_df_categorical.info()

### Решение

#### 1. Преобразование InvoiceNo: разделение на InvoiceNo (число) и InvoiceType (префикс)


In [ ]:
print("\n1. Преобразование InvoiceNo:")
print("-" * 60)

# Анализируем структуру InvoiceNo
print("Примеры значений InvoiceNo:")
sample_invoices = retail_df_categorical['InvoiceNo'].unique()[:20]
print(sample_invoices)

# Проверяем, есть ли значения с префиксами (не только цифры)
has_prefix = retail_df_categorical['InvoiceNo'].str.contains(r'^[A-Za-z]', regex=True, na=False)
print(f"\nКоличество значений с префиксом (буквы): {has_prefix.sum()}")
print(f"Примеры значений с префиксом:")
display(retail_df_categorical[has_prefix]['InvoiceNo'].unique()[:10])

# Разделяем InvoiceNo на числовую часть и префикс
def split_invoice_no(invoice_str):
    """Разделяет InvoiceNo на префикс и числовую часть"""
    if pd.isna(invoice_str):
        return None, None
    
    invoice_str = str(invoice_str)
    # Ищем префикс (все нецифровые символы в начале)
    match = re.match(r'^([A-Za-z]*)(\d+)$', invoice_str)
    if match:
        prefix = match.group(1) if match.group(1) else ''
        number = int(match.group(2))
        return prefix, number
    else:
        # Если не удалось распарсить, возвращаем как есть
        return '', int(invoice_str) if invoice_str.isdigit() else 0

# Применяем функцию разделения
invoice_parts = retail_df_categorical['InvoiceNo'].apply(split_invoice_no)

retail_df_categorical['InvoiceType'] = invoice_parts.apply(lambda x: x[0] if x[0] else '-')
retail_df_categorical['InvoiceNo_num'] = invoice_parts.apply(lambda x: x[1] if x[1] is not None else 0)

# Удаляем исходный столбец InvoiceNo и переименовываем новый
retail_df_categorical = retail_df_categorical.drop(columns=['InvoiceNo'])
retail_df_categorical = retail_df_categorical.rename(columns={'InvoiceNo_num': 'InvoiceNo'})

# Преобразуем InvoiceNo в int
retail_df_categorical['InvoiceNo'] = retail_df_categorical['InvoiceNo'].astype(int)

print("\nРезультат преобразования:")
print(f"Уникальных типов InvoiceType: {retail_df_categorical['InvoiceType'].nunique()}")
print(f"Распределение InvoiceType:")
value_counts = retail_df_categorical['InvoiceType'].value_counts()

print("\nПримеры данных после преобразования:")
for invoice_type in value_counts.index:
    print(f"примеры с типом : {invoice_type}")
    mask = retail_df_categorical['InvoiceType'] == invoice_type
    display(retail_df_categorical[mask][['InvoiceNo', 'InvoiceType']].head())


Дополнительно: кодируем InvoiceType с помощью Label Encoding

In [ ]:
invoice_type_encoder = LabelEncoder()
retail_df_categorical['InvoiceType_encoded'] = invoice_type_encoder.fit_transform(retail_df_categorical['InvoiceType'])

# Заменяем исходный InvoiceType на закодированный
retail_df_categorical = retail_df_categorical.drop(columns=['InvoiceType'])
retail_df_categorical = retail_df_categorical.rename(columns={'InvoiceType_encoded': 'InvoiceType'})

print("InvoiceType закодирован:")
print(f"Соответствие исходных значений и кодов:")
invoice_type_mapping = pd.DataFrame({
    'Исходное значение': invoice_type_encoder.classes_,
    'Код': range(len(invoice_type_encoder.classes_))
})
display(invoice_type_mapping)


In [ ]:
value_counts = retail_df_categorical['InvoiceType'].value_counts()

print("\nПримеры данных после преобразования:")
for invoice_type in value_counts.index:
    print(f"примеры с типом : {invoice_type}")
    mask = retail_df_categorical['InvoiceType'] == invoice_type
    display(retail_df_categorical[mask][['InvoiceNo', 'InvoiceType']].head())

#### 2. Преобразование StockCode: Label Encoding


In [ ]:
# Создаем кодировщик
stockcode_encoder = LabelEncoder()

# Применяем Label Encoding
retail_df_categorical['StockCode_encoded'] = stockcode_encoder.fit_transform(retail_df_categorical['StockCode'])

# Сохраняем исходный StockCode для справки (можно удалить позже)
print(f"Уникальных значений StockCode: {retail_df_categorical['StockCode'].nunique()}")
print(f"Диапазон закодированных значений: {retail_df_categorical['StockCode_encoded'].min()} - {retail_df_categorical['StockCode_encoded'].max()}")

print("\nПримеры преобразования:")
sample_stock = retail_df_categorical[['StockCode', 'StockCode_encoded']].head(10)
display(sample_stock)

# Заменяем исходный столбец на закодированный
retail_df_categorical = retail_df_categorical.drop(columns=['StockCode'])
retail_df_categorical = retail_df_categorical.rename(columns={'StockCode_encoded': 'StockCode'})


#### 3. Преобразование Description: хеширование


Обоснование использования хеширования:
- Description содержит текстовые описания товаров
- Большое количество уникальных значений (тысячи)
- Текстовые данные занимают много памяти
- Хеширование позволяет преобразовать строки в фиксированный размер (число)
- Сохраняет возможность группировки по одинаковым описаниям
- Уменьшает размер датасета в памяти

In [ ]:
# Анализируем Description перед преобразованием
print(f"Уникальных значений Description: {retail_df_categorical['Description'].nunique()}")
print(f"Общее количество строк: {len(retail_df_categorical)}")
print(f"Средняя длина описания: {retail_df_categorical['Description'].str.len().mean():.1f} символов")


# Используем встроенную функцию hash() Python для хеширования
# Применяем хеширование к каждому описанию
retail_df_categorical['Description_hash'] = retail_df_categorical['Description'].apply(
    lambda x: hash(str(x)) if pd.notna(x) else hash('')
)

# Преобразуем в положительные числа (модуль хеша)
retail_df_categorical['Description_hash'] = retail_df_categorical['Description_hash'].abs()

# Проверяем коллизии (одинаковые хеши для разных описаний)
original_unique = retail_df_categorical['Description'].nunique()
hash_unique = retail_df_categorical['Description_hash'].nunique()
collisions = original_unique - hash_unique

print(f"Уникальных описаний: {original_unique}")
print(f"Уникальных хешей: {hash_unique}")
print(f"Коллизий (если есть): {collisions}")

# Заменяем исходный столбец на хешированный
retail_df_categorical = retail_df_categorical.drop(columns=['Description'])
retail_df_categorical = retail_df_categorical.rename(columns={'Description_hash': 'Description'})

print("\nПримеры преобразования (первые 10 строк):")
# Показываем примеры (но уже без исходного Description, так как он удален)
display(retail_df_categorical[['Description']].head(10))


#### 4. Преобразование Country: Label Encoding


In [ ]:
# Создаем кодировщик для Country
from matplotlib.axes import Axes


country_encoder = LabelEncoder()

# Применяем Label Encoding
retail_df_categorical['Country_encoded'] = country_encoder.fit_transform(retail_df_categorical['Country'])

print(f"Уникальных значений Country: {retail_df_categorical['Country'].nunique()}")
print(f"Диапазон закодированных значений: {retail_df_categorical['Country_encoded'].min()} - {retail_df_categorical['Country_encoded'].max()}")

# Показываем соответствие между исходными значениями и кодами
country_mapping = pd.DataFrame({
    'Country': country_encoder.classes_,
    'Code': range(len(country_encoder.classes_))
})
print("\nСоответствие Country -> Code:")
display(country_mapping)

print("\nРаспределение по странам (топ-10):")
country_counts = retail_df_categorical['Country'].value_counts().head(10)
display(country_counts)

# Заменяем исходный столбец на закодированный
retail_df_categorical = retail_df_categorical.drop(columns=['Country'])
retail_df_categorical = retail_df_categorical.rename(columns={'Country_encoded': 'Country'})


#### Итоговая проверка преобразований


In [ ]:
print("\nИнформация о датасете после всех преобразований:")

comparison_types = pd.DataFrame({
    'Столбец': retail_df_categorical.columns,
    'Тип данных': [str(dtype) for dtype in retail_df_categorical.dtypes]
})
display(comparison_types)

print("\nПервые 10 строк преобразованного датасета:")
display(retail_df_categorical.head(10))

print("\nСтатистика по числовым признакам:")
display(retail_df_categorical.describe())

print("\n✓ Все преобразования категориальных данных выполнены успешно!")


## Задание 5

### Формулировка

* К временным данным следует применить квантование.

### Решение

В датасете соответствующему варианту #4 нет временн**ы**х данных, к которым имело бы смысл применять квантование.